# 06 - Brain MRI: Evaluation and Visualisation


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


**Goal:** measure the trained U-Net on the held-out test patients and show what
its predictions actually look like - including the failures.

Protocol:

1. sweep the binarisation threshold on **validation**, pick the best mean Dice;
2. apply that frozen threshold to **test**;
3. per-slice Dice, IoU/Jaccard, sensitivity and specificity;
4. bootstrap confidence intervals, resampled **by patient** (slices from one
   volume are not independent);
5. qualitative overlays for the best *and* worst cases.

**Prerequisite:** the checkpoint from notebook 05.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.common import get_device, load_config, seed_everything
from src.common.errors import DataNotFoundError
from src.common.io_utils import save_csv, save_figure
from src.common.viz import set_plot_style
from src.segmentation import build_seg_dataloaders, build_segmentation_model, load_checkpoint
from src.segmentation.evaluate import evaluate_segmentation

set_plot_style()
pd.set_option("display.width", 150)

cfg = load_config("mri_unet.yaml")
seed_everything(cfg.get("seed", 42), deterministic=cfg.get("deterministic", True))

pairs = pd.read_csv(PROJECT_ROOT / "data/processed/mri_pairs.csv")
device = get_device("auto")

run_name = cfg.get("run_name")
CKPT = PROJECT_ROOT / cfg.get("output.root") / run_name / "models" / f"{run_name}_best.pt"
if not CKPT.exists():
    raise DataNotFoundError(
        f"Checkpoint not found: {CKPT}\n"
        "  Fix: run 05_mri_unet_training.ipynb first."
    )

model = load_checkpoint(CKPT, build_segmentation_model(cfg), device)
loaders = build_seg_dataloaders(pairs, cfg, splits=("val", "test"))

## 1. Full evaluation

`evaluate_segmentation` runs the whole protocol and writes to
`outputs/segmentation/<run>/`:

| file | content |
|---|---|
| `metrics/threshold_sweep_validation.csv` | Dice vs threshold on validation |
| `metrics/per_slice_metrics_test.csv` | Dice, IoU, sensitivity per test slice |
| `metrics/metrics_test.json` | summary + bootstrap confidence intervals |
| `figures/overlays_best.png`, `overlays_worst.png` | qualitative results |
| `predictions/masks/*.png` | predicted binary masks |

In [ ]:
results = evaluate_segmentation(model, loaders, cfg, device, pair_table=pairs)

## 2. Headline metrics

**Read `mean_dice_tumour_slices`, not `mean_dice_all_slices`, as the primary
result.** Empty-mask slices where the model correctly predicts nothing score a
Dice of 1.0 by convention, so a dataset full of empty slices inflates the
all-slices average. Both are reported here; state clearly which one you quote.

`global_dice` pools all pixels across the test set - it is dominated by large
tumours and is included only for completeness.

In [ ]:
summary = results["summary"]
headline = pd.DataFrame([
    {"metric": "Mean Dice (tumour slices)  <- primary", "value": summary["mean_dice_tumour_slices"]},
    {"metric": "Mean IoU / Jaccard (tumour slices)", "value": summary["mean_iou_tumour_slices"]},
    {"metric": "Median Dice (tumour slices)", "value": summary["median_dice_tumour_slices"]},
    {"metric": "Mean sensitivity (tumour slices)", "value": summary["mean_sensitivity_tumour_slices"]},
    {"metric": "Mean specificity (all slices)", "value": summary["mean_specificity_all_slices"]},
    {"metric": "Mean Dice (all slices, empty=1.0)", "value": summary["mean_dice_all_slices"]},
    {"metric": "Global Dice (pixels pooled)", "value": summary["global_dice"]},
    {"metric": "Binarisation threshold (from validation)", "value": summary["threshold"]},
    {"metric": "Test slices", "value": summary["n_slices"]},
    {"metric": "Test slices containing tumour", "value": summary["n_slices_with_tumour"]},
])
save_csv(headline, results["dirs"]["metrics"] / "headline_metrics.csv")
headline

### Confidence intervals

Bootstrap resampled **by patient** where patient IDs are available - resampling by slice would treat near-identical neighbouring slices as independent evidence and produce intervals that are too narrow.

In [ ]:
for key in ["dice_ci_tumour_slices", "iou_ci_tumour_slices"]:
    ci = summary.get(key)
    if ci and "ci_low" in ci:
        print(f"{ci['metric']:<6s} {ci['point_estimate']:.4f} "
              f"[{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]  "
              f"(95% CI, resampled by {ci['resampling_unit']})")
    else:
        print(f"{key}: not computed (eval.n_bootstrap = 0)")

## 3. Distribution of per-slice Dice

The mean hides the spread. A bimodal distribution - many slices near 0.85 and a cluster near 0 - is common and much more informative than the average alone. The near-zero cases are usually very small tumours.

In [ ]:
per_slice = results["per_slice"]
print(per_slice["dice"].describe().round(4).to_string())

tumour_only = per_slice[per_slice["has_tumour"] == 1]
print(f"\nTumour slices with Dice < 0.10 (essentially missed): "
      f"{(tumour_only['dice'] < 0.10).sum()} / {len(tumour_only)}")
print(f"Tumour slices with Dice > 0.80: {(tumour_only['dice'] > 0.80).sum()} / {len(tumour_only)}")

### Does tumour size explain the failures?

Small lesions are systematically harder: a few misplaced pixels cost a large fraction of a small mask. If your worst cases are the smallest tumours, say so - it is a real and reportable limitation.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(tumour_only["true_pixels"], tumour_only["dice"], alpha=0.5, s=18, color="#4C72B0")
ax.set_xscale("log")
ax.set_xlabel("Ground-truth tumour size (pixels, log scale)")
ax.set_ylabel("Per-slice Dice")
ax.set_title("Segmentation quality vs tumour size")
save_figure(fig, results["dirs"]["figures"] / "dice_vs_tumour_size.png", close=False)

bins = pd.cut(tumour_only["true_pixels"], bins=[0, 100, 500, 2000, 10000, np.inf],
              labels=["<100", "100-500", "500-2k", "2k-10k", ">10k"])
print(tumour_only.groupby(bins, observed=True)["dice"].agg(["count", "mean"]).round(3).to_string())

## 4. Per-patient results

Aggregating by patient shows whether errors are concentrated in a few difficult cases or spread evenly - a distinction the slice-level mean cannot make.

In [ ]:
if "patient_id" in per_slice.columns and per_slice["patient_id"].notna().any():
    by_patient = (per_slice[per_slice["has_tumour"] == 1]
                  .groupby("patient_id")
                  .agg(n_slices=("dice", "size"), mean_dice=("dice", "mean"),
                       mean_iou=("iou", "mean"))
                  .sort_values("mean_dice"))
    save_csv(by_patient.reset_index(), results["dirs"]["metrics"] / "per_patient_metrics_test.csv")
    print(f"{len(by_patient)} test patients with tumour slices\n")
    print("Worst 5 patients:")
    print(by_patient.head(5).round(4).to_string())
    print("\nBest 5 patients:")
    print(by_patient.tail(5).round(4).to_string())
else:
    print("[info] no patient IDs available - per-patient breakdown skipped.")

## 5. Qualitative results

Both the best and the worst cases, saved to
`outputs/segmentation/<run>/figures/`. In the fourth column:
**green = correctly segmented tumour, red = false positive, blue = missed tumour**.

Showing the worst cases is not optional. A figure containing only successful
segmentations misrepresents the model.

In [ ]:
from IPython.display import Image as IPyImage, display

for name in ["best", "worst"]:
    path = results["dirs"]["figures"] / f"overlays_{name}.png"
    if path.exists():
        print(f"--- {name.upper()} cases ---")
        display(IPyImage(filename=str(path)))

### Middle-of-the-distribution cases

Best and worst are the extremes. Typical cases are what the mean Dice actually represents, so include one of these in the report too.

In [ ]:
from src.segmentation.evaluate import plot_prediction_overlays

tumour_idx = tumour_only.sort_values("dice").index.tolist()
if len(tumour_idx) >= 3:
    middle = tumour_idx[len(tumour_idx) // 2 - 1: len(tumour_idx) // 2 + 2]
    fig = plot_prediction_overlays(
        [results["images"][i] for i in middle],
        [results["truths"][i] for i in middle],
        [results["probabilities"][i] for i in middle],
        threshold=results["threshold"],
        titles=[f"Dice={per_slice.iloc[i]['dice']:.3f}" for i in middle],
        suptitle="Typical (median) test predictions",
    )
    save_figure(fig, results["dirs"]["figures"] / "overlays_typical.png", close=False)

## 6. Threshold sensitivity

How much the result depends on the binarisation threshold. A flat curve means the model is well calibrated; a sharp peak means the reported Dice is fragile and depends heavily on a value tuned on a small validation set.

In [ ]:
sweep = results.get("threshold_sweep")
if sweep is not None:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(sweep["threshold"], sweep["mean_dice"], marker="o", lw=2, color="#4C72B0")
    ax.axvline(results["threshold"], color="red", ls="--",
               label=f"chosen = {results['threshold']:.2f}")
    ax.set_xlabel("Binarisation threshold")
    ax.set_ylabel("Mean Dice (validation)")
    ax.set_title("Threshold sweep on the validation set")
    ax.legend()
    save_figure(fig, results["dirs"]["figures"] / "threshold_sweep.png", close=False)
    print(sweep.to_string(index=False))

---

## Notes for the report

Write these from the numbers above:

- mean Dice and IoU on tumour-containing test slices, **with** confidence
  intervals, and state the threshold and how it was chosen;
- the split of slices that were segmented well vs essentially missed;
- the relationship between tumour size and Dice;
- what the worst-case overlays show (over-segmentation? missed small lesions?
  false positives in healthy tissue?).

**Limitations to state explicitly:**

- 2D slice-wise segmentation, no 3D volumetric consistency between slices;
- a single public dataset, single annotation source, no inter-rater comparison;
- no external validation and no clinical validation whatsoever;
- Dice is an overlap statistic, not a measure of clinical usefulness - a mask
  with good Dice can still be unusable for surgical planning.

### Screenshots for the report
- The headline metrics table with confidence intervals
- Dice distribution histogram
- Best-case and worst-case overlays (both)
- Dice vs tumour size plot

---

## Project complete

All six analysis notebooks have been run. See `docs/SCREENSHOT_CHECKLIST.md`
for the full list of figures to collect for the final report and presentation.